# 01 - Data understanding & cleaning
**Business question:** can we trust the data enough to measure customer behaviour?

The dataset, its source and kind (real / simulated) are shown in the banner of each notebook and described in the README. For the bundled synthetic demo, realistic quality problems were injected on purpose so that the cleaning decisions below are real decisions.

In [1]:
import sys, json, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
from src.config import *
from src import eda, evaluation as ev
from src.feature_engineering import *
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

In [2]:
from src.adapter import load_config, standardize, mapping_table
from src.data_processing import inspect_data, clean_data
meta = json.load(open(DATA_PROC / "run_metadata.json"))
cfg = load_config(meta["config"])
print(f"Dataset: {meta['name']} ({'SIMULATED' if meta['kind'] == 'synthetic' else 'real'})")
display(mapping_table(cfg))                                   # original column -> canonical column
cust_raw, tx_raw, av, notes = standardize(cfg, save=False)    # adapter: raw file(s) -> canonical schema
print(*notes, sep="\n")
rep = inspect_data(cust_raw, tx_raw, cfg)
display(rep["shapes"]); display(rep["transactions_profile"]); display(rep["customers_profile"])
display(rep["issues"].to_frame())

Dataset: online_retail_ii (real)


,original_column,canonical_column,table
0,Customer ID,customer_id,transactions
1,Invoice,order_id,transactions
2,(unavailable),transaction_id,transactions
3,InvoiceDate,date,transactions
4,StockCode,product_id,transactions
5,Description,product_name,transactions
6,(unavailable),category,transactions
7,Quantity,quantity,transactions
8,Price,unit_price,transactions
9,(unavailable),discount,transactions


discount not mapped: set to 0 meaning 'discount information unavailable' (customers may still have received discounts).
Return rule 'id_prefix': 19,493 of 1,067,371 rows flagged as returns/cancellations.
5,830 lines removed because their product id is a configured non-product code (postage, fees, adjustments...).
No customer table: customers derived from transactions; demographics unavailable.


,table,rows,columns
0,customers,5876,5
1,transactions,1061541,12


,dtype,missing,missing_pct,n_unique
transaction_id,string,0,0.00,1061541
order_id,string,0,0.00,52179
customer_id,string,240913,22.69,5876
date,datetime64[us],0,0.00,46452
product_id,string,0,0.00,5288
product_name,string,4369,0.41,5639
category,string,1061541,100.00,0
quantity,int64,0,0.00,1055
unit_price,float64,0,0.00,718
discount,float64,0,0.00,1


,dtype,missing,missing_pct,n_unique
customer_id,string,0,0.0,5876
customer_registration_date,float64,5876,100.0,0
customer_age,float64,5876,100.0,0
customer_gender,float64,5876,100.0,0
customer_location,float64,5876,100.0,0


,count
rows without customer id,240913
exact duplicate rows,34223
dates that could not be parsed,0
returns / cancellations (per configured rule),18286
negative quantity NOT flagged as return,3457
zero or missing quantity,0
unit_price <= 0 or missing,6174
"discount outside [0, 1]",0
impossible ages (<16 or >100),0
missing ages,5876


## Investigate before deleting
Unusual values are not automatically errors. We look at each one first.

In [3]:
neg = tx_raw[tx_raw.quantity < 0].copy()
neg["kind"] = np.where(neg.is_return, "flagged as return by the configured rule", "NOT flagged (treated as invalid)")
display(neg.kind.value_counts().to_frame())
print("return rule:", av["return_rule"])
display(neg[~neg.is_return].head())

,count
kind,
flagged as return by the configured rule,18286
NOT flagged (treated as invalid),3457


return rule: {'method': 'id_prefix', 'detected_by': None, 'prefixes': ['C'], 'method_used': 'id_prefix'}


,transaction_id,order_id,customer_id,date,product_id,product_name,category,quantity,unit_price,discount,payment_method,is_return,kind
260,L00000263,489464,<NA>,2009-12-01 10:52:00,21733,85123a mixed,<NA>,-96,0.0,0.0,<NA>,False,NOT flagged (treated as invalid)
280,L00000283,489463,<NA>,2009-12-01 10:52:00,71477,short,<NA>,-240,0.0,0.0,<NA>,False,NOT flagged (treated as invalid)
281,L00000284,489467,<NA>,2009-12-01 10:53:00,85123A,21733 mixed,<NA>,-192,0.0,0.0,<NA>,False,NOT flagged (treated as invalid)
467,L00000470,489521,<NA>,2009-12-01 11:44:00,21646,<NA>,<NA>,-50,0.0,0.0,<NA>,False,NOT flagged (treated as invalid)
3092,L00003114,489655,<NA>,2009-12-01 17:26:00,20683,<NA>,<NA>,-44,0.0,0.0,<NA>,False,NOT flagged (treated as invalid)


In [4]:
bad = tx_raw[tx_raw.unit_price.fillna(0) <= 0].copy()
print(f"{len(bad):,} rows with price <= 0 or missing; configured repair: {cfg['cleaning']['nonpositive_price']}")
if av["product_id"] and len(bad):
    med = tx_raw[tx_raw.unit_price > 0].groupby("product_id").unit_price.agg(["median", "std"])
    print("Within-product price spread (std) - a small value means one list price per product, so the product median is a sound repair:")
    display(med["std"].describe().round(3).to_frame().T)

6,174 rows with price <= 0 or missing; configured repair: drop
Within-product price spread (std) - a small value means one list price per product, so the product median is a sound repair:


,count,mean,std,min,25%,50%,75%,max
std,4763.0,1.298,3.862,0.0,0.256,0.685,1.427,185.355


## Cleaning (every decision is logged)

In [5]:
cust, tx_clean, log, summ = clean_data(cust_raw, tx_raw, cfg, av)
display(log)
print(f"rows: {summ['rows_in']:,} in -> {summ['rows_out']:,} kept | purchase lines {summ['normal_lines_kept']:,} | returns kept {summ['return_lines_kept']:,}")
print("removed by reason:", summ["removed_by_reason"])
print(f"customers in table: {len(cust_raw):,} -> {len(cust):,}")

,issue,rows_affected,decision,rationale
0,Dates in several formats,0,All ISO / native dates,Time of day is dropped. Ambiguous formats shou...
1,Rows without customer id,240913,Dropped (reported),"Cannot be attributed to a customer, so they ca..."
2,Exact duplicate rows,26405,Dropped (kept first),Identical in every field: double-posted record...
3,Returns / cancellations (per configured rule),17586,"Kept, flagged is_return=True",Returns reduce net revenue but are NOT purchas...
4,"Invalid quantity (zero, missing, or negative a...",0,Dropped,0 rows are negative but not flagged as returns...
5,unit_price <= 0 or missing,60,Dropped,"Zero/negative prices are free items, adjustmen..."
6,Unusually large line values (> 99.9th percentile),777,"Kept, flagged suspicious_value",Large baskets are plausible; they are real rev...
7,Transactions of customers missing from the cus...,0,None found,
8,Registration date unavailable,5876,Approximated by the first observed purchase date,APPROXIMATION: customers may have registered e...


rows: 1,061,541 in -> 794,163 kept | purchase lines 776,577 | returns kept 17,586
removed by reason: {'missing customer id': 240913, 'exact duplicate row': 26405, 'invalid price (<= 0 or missing)': 60}
customers in table: 5,876 -> 5,876


### So what?
* Check the "removed by reason" line above: it states exactly how many rows were dropped and why. Anything that can be repaired from other information (dates, spellings, units) is repaired; only unverifiable rows are dropped.
* Returns / cancellations are kept and flagged by the **configured** rule: they matter for *net* revenue but are **not purchases** and never count as customer activity in the churn definition. If the "NOT flagged" count above is large, the return rule in the config is probably wrong.
* Cleaned tables are the only input to every later notebook.